# Stage 1C — PIT Revenue Growth

This notebook freezes and validates two raw Growth components from immutable SEC evidence. It constructs neither a scalar Growth score nor `X_char`.


## Frozen definition

`GrowthMagnitude = (Revenue_TTM_PIT(t) / Revenue_TTM_PIT(t-3y))**(1/3) - 1` and `GrowthPersistence = mean(g1 > 0, g2 > 0, g3 > 0)`. Exactly four valid, comparable annual PIT Revenue TTM points are required; otherwise both values are `NA`.


In [ ]:
from __future__ import annotations
import json
from pathlib import Path
import sys
import numpy as np
import pandas as pd

FORMS = {'10-Q', '10-K', '10-Q/A', '10-K/A'}
RAW_ROOT = next(p / 'data' / 'raw' / 'stage_0b_sec' for p in [Path.cwd(), *Path.cwd().parents] if (p / 'data' / 'raw' / 'stage_0b_sec' / 'manifest.json').exists())
PROJECT_ROOT = RAW_ROOT.parents[2]
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from pit_revenue_growth import REVENUE_TAGS, revenue_growth_as_of, revenue_ttm_as_of

manifest = json.loads((RAW_ROOT / 'manifest.json').read_text(encoding='utf-8'))
def read_raw(entry):
    return json.loads((PROJECT_ROOT / entry['local_raw_file']).read_text(encoding='utf-8'))

payloads, filing_rows, seen = {}, [], set()
resolved = {x['issuer']: x['resolved_cik'] for x in manifest if x['endpoint'] == 'identity_resolution'}
for item in manifest:
    if item['request_status'] != 'HTTP_200' or not item.get('local_raw_file'):
        continue
    if item['endpoint'] == 'companyfacts':
        payload = read_raw(item)
        if str(payload.get('cik', '')).zfill(10) == resolved.get(item['issuer'], str(payload.get('cik', '')).zfill(10)):
            payloads[item['issuer']] = payload
    if item['endpoint'] in {'submissions', 'submissions_history'} and item['content_sha256'] not in seen:
        seen.add(item['content_sha256'])
        inventory = read_raw(item).get('filings', {}).get('recent', read_raw(item))
        for values in zip(*inventory.values()):
            row = dict(zip(inventory, values))
            if row.get('form') in FORMS:
                filing_rows.append({'issuer': item['issuer'], **row})

filings = pd.DataFrame(filing_rows).drop_duplicates(['issuer', 'accessionNumber'])[['issuer', 'accessionNumber', 'form', 'filingDate', 'acceptanceDateTime']]
filings['acceptanceDateTime'] = pd.to_datetime(filings['acceptanceDateTime'], utc=True)
print(f'Reading immutable SEC evidence: {RAW_ROOT.relative_to(PROJECT_ROOT)}')


In [ ]:
revenue_rows = []
for issuer, tag in REVENUE_TAGS.items():
    payload = payloads[issuer]
    cik = str(payload['cik']).zfill(10)
    for unit, observations in payload.get('facts', {}).get('us-gaap', {}).get(tag, {}).get('units', {}).items():
        for fact in observations:
            if fact.get('form') in FORMS:
                revenue_rows.append({'issuer': issuer, 'cik': cik, 'tag': tag, 'unit': unit, 'start': fact.get('start'), 'end': fact.get('end'), 'value': fact.get('val'), 'fy': fact.get('fy'), 'fp': fact.get('fp'), 'form': fact.get('form'), 'accn': fact.get('accn'), 'frame': fact.get('frame')})
lookup = filings.rename(columns={'accessionNumber': 'accn', 'form': 'filing_form'})
revenue_facts = pd.DataFrame(revenue_rows).merge(lookup, on=['issuer', 'accn'], how='left', validate='many_to_one')
assert revenue_facts['acceptanceDateTime'].notna().all(), 'each fact requires source-filing acceptance provenance'
revenue_facts[['issuer', 'tag', 'unit', 'form', 'accn', 'acceptanceDateTime']].head()


## PIT TTM with provenance

The engine chooses compatible direct or cumulative facts only from the information set eligible at the supplied decision time. The shown rows preserve accessions, forms, acceptance timestamps, eligibility, and construction method.


In [ ]:
unh_decision = '2026-03-02T14:30:00Z'
unh_ttm = revenue_ttm_as_of(revenue_facts, 'UNH', unh_decision)
assert unh_ttm.status == 'VALID'
unh_ttm.quarters[['economic_quarter_start', 'economic_quarter_end', 'value', 'method', 'source_accessions', 'source_forms', 'source_acceptance_datetimes', 'pit_eligibility', 'revenue_tag']]


## Frozen calculation: magnitude and persistence remain distinct


In [ ]:
def display_growth(issuer, decision_time):
    result = revenue_growth_as_of(revenue_facts, issuer, decision_time)
    points = [result.revenue_ttm_t3, result.revenue_ttm_t2, result.revenue_ttm_t1, result.revenue_ttm_t0]
    return result, pd.DataFrame({
        'annual_point': ['t-3y', 't-2y', 't-1y', 't'],
        'Revenue_TTM_PIT': [None if x is None else x.value for x in points],
        'TTM_status': [None if x is None else x.status for x in points],
        'quarter_accessions': [None if x is None else tuple(x.quarters['source_accessions']) for x in points],
    })

unh_growth, unh_points = display_growth('UNH', unh_decision)
assert unh_growth.status == 'VALID'
manual_cagr = (unh_growth.revenue_ttm_t0.value / unh_growth.revenue_ttm_t3.value)**(1/3) - 1
manual_persistence = np.mean(np.array([unh_growth.annual_growth_1, unh_growth.annual_growth_2, unh_growth.annual_growth_3]) > 0)
assert np.isclose(unh_growth.growth_magnitude, manual_cagr)
assert np.isclose(unh_growth.growth_persistence, manual_persistence)
print({'GrowthMagnitude': unh_growth.growth_magnitude, 'GrowthPersistence': unh_growth.growth_persistence, 'manual_CAGR': manual_cagr, 'manual_persistence': manual_persistence})
unh_points


AAPL provides the useful contrast: an intermediate contraction can coexist with a positive multi-year magnitude, so the two outputs must not be collapsed.


In [ ]:
aapl_growth, aapl_points = display_growth('AAPL', '2025-11-03T14:30:00Z')
assert aapl_growth.status == 'VALID'
print({'GrowthMagnitude': aapl_growth.growth_magnitude, 'GrowthPersistence': aapl_growth.growth_persistence, 'annual_intervals': [aapl_growth.annual_growth_1, aapl_growth.annual_growth_2, aapl_growth.annual_growth_3]})
aapl_points


## Cross-issuer validation and explicit NA

This table validates only the accounting/PIT construction. It does not rank issuers or assert identical economic meaning across sectors; JPM remains a banking-revenue representation.


In [ ]:
decisions = {'AAPL': '2025-11-03T14:30:00Z', 'JPM': '2026-02-17T14:30:00Z', 'XOM': '2026-02-19T14:30:00Z', 'UNH': '2026-03-02T14:30:00Z', 'WMT': '2026-03-16T14:30:00Z'}
rows = []
for issuer, decision in decisions.items():
    r = revenue_growth_as_of(revenue_facts, issuer, decision)
    rows.append({'issuer': issuer, 'decision_time': r.decision_time, 'revenue_tag': REVENUE_TAGS[issuer], 'status': r.status, 'GrowthMagnitude': r.growth_magnitude, 'GrowthPersistence': r.growth_persistence, 'g1': r.annual_growth_1, 'g2': r.annual_growth_2, 'g3': r.annual_growth_3, 'reason': r.reason})
cross_issuer = pd.DataFrame(rows)
cross_issuer


In [ ]:
# Before four annual Revenue TTM observations exist, the result stays NA: no shorter-horizon substitute.
insufficient = revenue_growth_as_of(revenue_facts, 'AAPL', '2019-11-01T14:30:00Z')
assert insufficient.status == 'NA_FEWER_THAN_FOUR_ANNUAL_TTMS'
assert insufficient.growth_magnitude is None and insufficient.growth_persistence is None
{'status': insufficient.status, 'reason': insufficient.reason, 'GrowthMagnitude': insufficient.growth_magnitude, 'GrowthPersistence': insufficient.growth_persistence}


## Decision and boundary

The frozen repository returns the two raw components with provenance or an explicit `NA`; it never shortens the three-year horizon, imputes, or splices a revenue taxonomy. Amendments are prospective through the underlying PIT engine. This notebook does not normalize/rank the results, combine them to a scalar Growth measure, construct `X_char`, `w_raw`, `F_risk`, portfolios, or backtests.
